In [1]:
import math
import re
from collections import Counter

import pandas as pd

df = pd.read_csv("imdb_processed.csv")


df["Description"] = df["Description"].fillna("")
df["Movie Name"] = df["Movie Name"].fillna("")


df["Movie Name Clean"] = df["Movie Name"].str.replace(r"^\s*\d+\.\s*", "", regex=True)

stopwords = {
    "the",
    "a",
    "an",
    "and",
    "or",
    "of",
    "to",
    "in",
    "on",
    "for",
    "with",
    "by",
    "from",
    "their",
    "his",
    "her",
    "its",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "as",
    "at",
    "that",
    "this",
    "these",
    "those",
    "it",
    "they",
    "them",
    "he",
    "she",
    "you",
    "your",
    "our",
    "us",
}



In [2]:
df.head()

,Movie Name,Duration,IMdb rating,Rating,Description,Votes,Poster,Month of Release,Movie Name Clean
0,1. Vindhya Victim Verdict V3,2h 7m,7.6,595,V3 is a fight for their rights and against an ...,595,https://m.media-amazon.com/images/M/MV5BYTE5Yj...,January,Vindhya Victim Verdict V3
1,2. Thunivu,2h 26m,6.1,29K,A mysterious mastermind - Daredevil and his te...,"28,737",https://m.media-amazon.com/images/M/MV5BY2M0OD...,January,Thunivu
2,3. Varisu,2h 49m,6.0,37K,Vijay Rajendran is a happy to-go lucky man. Th...,"36,755",https://m.media-amazon.com/images/M/MV5BMjE4Yj...,January,Varisu
3,4. Oangaram,1h 52m,NaN,NaN,,NaN,NaN,January,Oangaram
4,5. Vallavanukkum Vallavan,2h 47m,7.5,13,"Two conmen, who believe that money can solve a...",13,https://m.media-amazon.com/images/M/MV5BZGRkMG...,January,Vallavanukkum Vallavan


In [3]:

def tokenize(text):
    words = re.findall(r"[a-zA-Z0-9]+", str(text).lower())

    cleaned_words = [w for w in words if w not in stopwords]

    return cleaned_words

In [4]:
doc_tokens = df["Description"].map(tokenize)
doc_lengths = doc_tokens.map(len).replace(0, 1)

In [5]:
doc_tokens

0      [v3, fight, rights, against, injustice, gang, ...
1      [mysterious, mastermind, daredevil, team, form...
2      [vijay, rajendran, happy, go, lucky, man, thin...
3                                                     []
4      [two, conmen, who, believe, money, can, solve,...
                             ...                        
204    [when, wife, sends, him, rehab, alcohol, addic...
205    [1975, filmmaker, agrees, collaborate, film, g...
206    [daring, robbery, jewellery, showroom, kicksta...
207                                                   []
208    [kathir, s, wife, chaitra, mentally, affected,...
Name: Description, Length: 209, dtype: object

In [6]:
doc_lengths

0      24
1      15
2      17
3       1
4      20
       ..
204    24
205    11
206    13
207     1
208    28
Name: Description, Length: 209, dtype: int64

In [7]:
tf_docs = []

for tokens, length in zip(doc_tokens, doc_lengths):
    counts = Counter(tokens)

    tf = {}

    for term, count in counts.items():
        tf[term] = count / length

    tf_docs.append(tf)

In [8]:
tf_docs

[{'v3': 0.041666666666666664,
  'fight': 0.041666666666666664,
  'rights': 0.041666666666666664,
  'against': 0.041666666666666664,
  'injustice': 0.041666666666666664,
  'gang': 0.041666666666666664,
  'rape': 0.041666666666666664,
  'murder': 0.041666666666666664,
  'encountered': 0.041666666666666664,
  'case': 0.041666666666666664,
  'achieving': 0.041666666666666664,
  'poetic': 0.041666666666666664,
  'justice': 0.041666666666666664,
  'help': 0.041666666666666664,
  'system': 0.041666666666666664,
  'which': 0.041666666666666664,
  'fails': 0.041666666666666664,
  'initially': 0.041666666666666664,
  'due': 0.041666666666666664,
  'undue': 0.041666666666666664,
  'pressure': 0.041666666666666664,
  'representatives': 0.041666666666666664,
  'thy': 0.041666666666666664,
  'people': 0.041666666666666664},
 {'mysterious': 0.06666666666666667,
  'mastermind': 0.06666666666666667,
  'daredevil': 0.06666666666666667,
  'team': 0.06666666666666667,
  'forms': 0.06666666666666667,
  'pl

In [9]:

df_counts = Counter()

for tokens in doc_tokens:
    unique_terms = set(tokens)

    for term in unique_terms:
        df_counts[term] += 1

In [10]:
df_counts

Counter({'who': 55,
         'when': 36,
         's': 35,
         'life': 32,
         'him': 31,
         'has': 25,
         'man': 23,
         'family': 21,
         'will': 20,
         'love': 19,
         'but': 19,
         'how': 18,
         'police': 18,
         'gets': 16,
         'which': 15,
         'father': 15,
         'what': 15,
         'into': 15,
         'friends': 15,
         'about': 15,
         'time': 14,
         'one': 13,
         'after': 13,
         'young': 12,
         'lives': 12,
         'girl': 12,
         'up': 12,
         'officer': 12,
         'people': 11,
         'murder': 11,
         'against': 11,
         'money': 11,
         'find': 11,
         'two': 11,
         'other': 11,
         'out': 11,
         'help': 10,
         'between': 10,
         'not': 10,
         'around': 10,
         'can': 9,
         'parents': 9,
         'each': 9,
         'get': 9,
         'story': 9,
         'while': 9,
         'gang': 8,
 

In [11]:
N = len(df)

idf_scores = {}

for term, df_value in df_counts.items():
    idf_scores[term] = math.log(N / df_value)

In [12]:
idf_scores

{'gang': 3.262892710284975,
 'help': 3.039749158970765,
 'undue': 5.342334251964811,
 'people': 2.9444389791664403,
 'representatives': 5.342334251964811,
 'fight': 3.7328963395307104,
 'justice': 4.6491870714048655,
 'rights': 5.342334251964811,
 'due': 3.396424102909498,
 'system': 5.342334251964811,
 'murder': 2.9444389791664403,
 'rape': 5.342334251964811,
 'fails': 4.6491870714048655,
 'thy': 5.342334251964811,
 'initially': 4.6491870714048655,
 'pressure': 4.6491870714048655,
 'case': 3.262892710284975,
 'poetic': 5.342334251964811,
 'achieving': 5.342334251964811,
 'encountered': 5.342334251964811,
 'injustice': 5.342334251964811,
 'against': 2.9444389791664403,
 'v3': 5.342334251964811,
 'which': 2.634284050862601,
 'forms': 3.9560398908449206,
 'money': 2.9444389791664403,
 'mastermind': 5.342334251964811,
 'daredevil': 5.342334251964811,
 'find': 2.9444389791664403,
 'plan': 3.9560398908449206,
 'looted': 5.342334251964811,
 'commits': 5.342334251964811,
 'team': 5.3423342519

In [13]:
tfidf_docs = []

for tf in tf_docs:
    tfidf = {}

    for term, tf_value in tf.items():
        tfidf[term] = tf_value * idf_scores[term]

    tfidf_docs.append(tfidf)

In [14]:
tfidf_docs

[{'v3': 0.2225972604985338,
  'fight': 0.15553734748044626,
  'rights': 0.2225972604985338,
  'against': 0.12268495746526834,
  'injustice': 0.2225972604985338,
  'gang': 0.13595386292854061,
  'rape': 0.2225972604985338,
  'murder': 0.12268495746526834,
  'encountered': 0.2225972604985338,
  'case': 0.13595386292854061,
  'achieving': 0.2225972604985338,
  'poetic': 0.2225972604985338,
  'justice': 0.1937161279752027,
  'help': 0.1266562149571152,
  'system': 0.2225972604985338,
  'which': 0.10976183545260837,
  'fails': 0.1937161279752027,
  'initially': 0.1937161279752027,
  'due': 0.1415176709545624,
  'undue': 0.2225972604985338,
  'pressure': 0.1937161279752027,
  'representatives': 0.2225972604985338,
  'thy': 0.2225972604985338,
  'people': 0.12268495746526834},
 {'mysterious': 0.22642827352729986,
  'mastermind': 0.35615561679765406,
  'daredevil': 0.35615561679765406,
  'team': 0.35615561679765406,
  'forms': 0.2637359927229947,
  'plan': 0.2637359927229947,
  'commits': 0.35

In [15]:

def search_movies(query, top_k=5):
    q_tokens = tokenize(query)

    if not q_tokens:
        return []

    q_counts = Counter(q_tokens)

    q_len = len(q_tokens)

    q_tf = {}

    for term, count in q_counts.items():
        q_tf[term] = count / q_len

    q_tfidf = {}

    for term, tf_value in q_tf.items():
        idf = idf_scores.get(term, 0)

        q_tfidf[term] = tf_value * idf

    scores = []

    for idx, doc_vector in enumerate(tfidf_docs):
        score = 0.0

        for term, q_weight in q_tfidf.items():
            doc_weight = doc_vector.get(term, 0.0)

            score += q_weight * doc_weight

        if score > 0:
            scores.append((score, idx))

    scores.sort(reverse=True)

    results = []

    for score, idx in scores[:top_k]:
        results.append(
            {
                "movie": df.loc[idx, "Movie Name Clean"],
                "score": round(score, 4),
                "description": df.loc[idx, "Description"],
            }
        )

    return results

In [16]:

queries = [
    "bank heist money",
    "murder investigation",
    "love family",
    "alien invasion",
    "crime gangster",
]


for q in queries:
    print("\n" + "=" * 60)

    print("QUERY:", q)

    print("=" * 60)

    results = search_movies(q, top_k=3)

    for row in results:
        print("\nMovie:", row["movie"])
        print("Score:", row["score"])
        print("Description:")
        print(row["description"])



QUERY: bank heist money

Movie: Thunivu
Score: 1.3072
Description:
A mysterious mastermind - Daredevil and his team forms a plan and commits bank heist to find the corporate looted people's money.

Movie: Kasethan Kadavulada
Score: 0.3612
Description:
A young man, his cousin and a friend stealing money from his stingy stepmother.

Movie: Vallavanukkum Vallavan
Score: 0.289
Description:
Two conmen, who believe that money can solve all their problems, decide to extract money from wrongdoers. Trouble begins when a mad policeman crosses their way.

QUERY: murder investigation

Movie: Run Baby Run
Score: 1.0133
Description:
A carefree Sathya faces a race against time after becoming involved in a complex murder investigation.

Movie: Oongi Adicha Ondra Ton Weightu Da
Score: 0.8695
Description:
Crimes happening in the city and the investigation about the killer forms the crux of the remaining.

Movie: Kidugu
Score: 0.6193
Description:
A group of young nationalists murder a couple of policeme